# 🥉 Bronze Layer — Raw Ingestion
**LushProtein Analytics | Medallion Architecture**

This notebook ingests **all raw source files as-is** with zero transformation — preserving the original data exactly as received.

| Source | Files | Key |
|--------|-------|-----|
| Shopify Orders | `1_1` → `1_7` (7 yearly xlsx) | `ID` (order), `Customer: ID` |
| Product Master | `2_1` | `Variant SKU` |
| Discounts | `3_1` | `Name` (code) |
| Traffic / Sessions | `4_1` | composite |
| Recharge Orders | `5_1` combined | `recharge_order_id` |
| Recharge Checkout Items | `5_2` | `recharge_order_id + product_sku` |
| Recharge Reactivated Subs | `5_3` | `customer_id` |
| Recharge Churned Subs | `5_4` | `subscription_id` |
| Recharge Recurring Items | `5_5` | `recharge_order_id + product_sku` |


## 0. Setup & Paths

In [1]:
import pandas as pd
import glob
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

# ─── Base paths ───────────────────────────────────────────────────────────────
BASE = os.path.dirname(os.path.abspath('__file__'))   # notebook dir = project dir
# Override if needed:
# BASE = '/Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505'

BRONZE_DIR = os.path.join(BASE, 'medallion', 'bronze')
os.makedirs(BRONZE_DIR, exist_ok=True)

print(f"Base     : {BASE}")
print(f"Bronze   : {BRONZE_DIR}")


Base     : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505
Bronze   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/bronze


## 1. Shopify Orders (7 yearly files → single bronze table)

In [2]:
order_files = sorted(glob.glob(os.path.join(BASE, '1.customer_transaction', '1_*.xlsx')))
print(f"Found {len(order_files)} order files:")
for f in order_files:
    print(f"  {os.path.basename(f)}")


Found 7 order files:
  1_1.orders-2020_20260505.xlsx
  1_2.orders-2021_20260505.xlsx
  1_3.orders-2022_20260505.xlsx
  1_4.orders-2023_20260505.xlsx
  1_5.orders-2024_20260505.xlsx
  1_6.orders-2025_20260505.xlsx
  1_7.orders-2026_20260505.xlsx


In [3]:
# Ingest each yearly file, tag with source filename, then union
dfs = []
for f in order_files:
    df = pd.read_excel(f, dtype=str)          # read everything as str to preserve raw values
    df['_source_file'] = os.path.basename(f)   # provenance column
    dfs.append(df)
    print(f"{os.path.basename(f):45s} → {df.shape[0]:>6,} rows × {df.shape[1]} cols")

bronze_orders = pd.concat(dfs, ignore_index=True)
print(f"\nBronze Orders combined: {bronze_orders.shape[0]:,} rows × {bronze_orders.shape[1]} cols")


1_1.orders-2020_20260505.xlsx                 → 12,201 rows × 70 cols
1_2.orders-2021_20260505.xlsx                 → 33,196 rows × 70 cols
1_3.orders-2022_20260505.xlsx                 → 18,563 rows × 70 cols
1_4.orders-2023_20260505.xlsx                 → 12,962 rows × 70 cols
1_5.orders-2024_20260505.xlsx                 → 26,373 rows × 70 cols
1_6.orders-2025_20260505.xlsx                 → 40,932 rows × 70 cols
1_7.orders-2026_20260505.xlsx                 →  9,601 rows × 70 cols

Bronze Orders combined: 153,828 rows × 70 cols


In [4]:
# Quick sanity check
print("Sample (first 5 rows, selected cols):")
display(bronze_orders[['ID', 'Name', 'Processed At', 'Currency', 'Source',
                        'Customer: ID', 'Top Row', 'Line: Type', '_source_file']].head())
print("\nNull counts (top 10 columns by null%):")
null_pct = bronze_orders.isnull().mean().sort_values(ascending=False).head(10)
print(null_pct.apply(lambda x: f"{x:.1%}"))


Sample (first 5 rows, selected cols):


,ID,Name,Processed At,Currency,Source,Customer: ID,Top Row,Line: Type,_source_file
0,4992746586367,LPSG-9541,2020-12-31 03:27:50 +0800,SGD,17346527233,6423777902847,1,Line Item,1_1.orders-2020_20260505.xlsx
1,4992746586367,LPSG-9541,2020-12-31 03:27:50 +0800,SGD,17346527233,6423777902847,NaN,Line Item,1_1.orders-2020_20260505.xlsx
2,4992746586367,LPSG-9541,2020-12-31 03:27:50 +0800,SGD,17346527233,6423777902847,NaN,Line Item,1_1.orders-2020_20260505.xlsx
3,4992746586367,LPSG-9541,2020-12-31 03:27:50 +0800,SGD,17346527233,6423777902847,NaN,Line Item,1_1.orders-2020_20260505.xlsx
4,4992746586367,LPSG-9541,2020-12-31 03:27:50 +0800,SGD,17346527233,6423777902847,NaN,Shipping Line,1_1.orders-2020_20260505.xlsx



Null counts (top 10 columns by null%):
Browser: Ad URL                   100.0%
Purchase Order Number             100.0%
Price: Total Refund                99.6%
Line: Variant Compare At Price     98.7%
Cancelled At                       97.6%
Cancel: Reason                     97.6%
Browser: UTM Content               96.2%
Browser: UTM Campaign              93.7%
Browser: UTM Source                93.5%
Browser: UTM Medium                93.5%
dtype: object


In [5]:
# Save bronze parquet
bronze_orders.to_parquet(os.path.join(BRONZE_DIR, 'bronze_orders.parquet'), index=False)
print("✅ Saved: bronze_orders.parquet")


✅ Saved: bronze_orders.parquet


## 2. Product Master

In [6]:
bronze_products = pd.read_excel(
    os.path.join(BASE, '2.product_master', '2_1.products_master_20260505.xlsx'),
    dtype=str
)
bronze_products['_source_file'] = '2_1.products_master_20260505.xlsx'
print(f"Product Master: {bronze_products.shape[0]:,} rows × {bronze_products.shape[1]} cols")
display(bronze_products.head(3))
bronze_products.to_parquet(os.path.join(BRONZE_DIR, 'bronze_products.parquet'), index=False)
print("✅ Saved: bronze_products.parquet")


Product Master: 167 rows × 25 cols


,Handle,Title,Body (HTML),Vendor,Variant SKU,Variant Grams,Variant Price,Variant Compare At Price,Variant Barcode,Image Src,Gift Card,Variant Image,Variant Weight Unit,Cost per item,Status,Included / Singapore,Price / Singapore,Compare At Price / Singapore,Included / Hong Kong,Price / Hong Kong,Compare At Price / Hong Kong,Included / Malaysia,Price / Malaysia,Compare At Price / Malaysia,_source_file
0,lean-protein-peach-oolong-pre-order,LEAN PROTEIN PEACH OOLONG (PRE-ORDER),<p>Elevate your fitness journey with our new Lean Protein! Perfect for a hea...,SG,LEAN-POL-1KG-V1,1000,59.9,NaN,'0724999807825,https://cdn.shopify.com/s/files/1/0660/4895/0527/files/IMG_9244.jpg?v=177065...,False,https://cdn.shopify.com/s/files/1/0660/4895/0527/files/lean-peachoolong-shad...,kg,17.65,active,True,NaN,NaN,True,NaN,NaN,True,179,NaN,2_1.products_master_20260505.xlsx
1,lean-protein-peach-oolong-pre-order,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://cdn.shopify.com/s/files/1/0660/4895/0527/files/lean-peachoolong-shad...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2_1.products_master_20260505.xlsx
2,lean-protein-peach-oolong-pre-order,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://cdn.shopify.com/s/files/1/0660/4895/0527/files/IMG_9181_copy.jpg?v=1...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2_1.products_master_20260505.xlsx


✅ Saved: bronze_products.parquet


## 3. Discounts

In [7]:
bronze_discounts = pd.read_csv(
    os.path.join(BASE, '3.Discounts', '3_1.discounts_export_20260505.csv'),
    dtype=str
)
bronze_discounts['_source_file'] = '3_1.discounts_export_20260505.csv'
print(f"Discounts: {bronze_discounts.shape[0]:,} rows × {bronze_discounts.shape[1]} cols")
display(bronze_discounts.head(3))
bronze_discounts.to_parquet(os.path.join(BRONZE_DIR, 'bronze_discounts.parquet'), index=False)
print("✅ Saved: bronze_discounts.parquet")


Discounts: 367 rows × 18 cols


,Name,Value,Value Type,Type,Discount Class,Minimum Purchase Requirements,Combines with Order Discounts,Combines with Product Discounts,Combines with Shipping Discounts,Customer Selection,Context,Times Used In Total,Applies Once Per Customer,Usage Limit Per Code,Status,Start,End,_source_file
0,SG40,-40.0,percentage,Amount Off,order,NaN,NaN,NaN,NaN,all,all,5,NaN,NaN,Expired,2022-09-18 14:39:30 +0800,2022-10-13 10:08:02 +0800,3_1.discounts_export_20260505.csv
1,september,-58.0,percentage,Amount Off,order,NaN,NaN,NaN,NaN,all,all,72,NaN,NaN,Expired,2022-09-24 09:30:00 +0800,2022-10-01 12:00:00 +0800,3_1.discounts_export_20260505.csv
2,9f9abe1f833f,-20.0,percentage,Amount Off,order,10.0,NaN,NaN,NaN,all,all,0,NaN,1.0,Active,2022-09-24 02:06:29 +0800,NaN,3_1.discounts_export_20260505.csv


✅ Saved: bronze_discounts.parquet


## 4. Traffic / Sessions by Referrer

In [8]:
bronze_sessions = pd.read_csv(
    os.path.join(BASE, '4.Campaigns', '4_1.Sessions by referrer_20260505.csv'),
    dtype=str
)
bronze_sessions['_source_file'] = '4_1.Sessions by referrer_20260505.csv'
print(f"Sessions: {bronze_sessions.shape[0]:,} rows × {bronze_sessions.shape[1]} cols")
display(bronze_sessions.head(3))
bronze_sessions.to_parquet(os.path.join(BRONZE_DIR, 'bronze_sessions.parquet'), index=False)
print("✅ Saved: bronze_sessions.parquet")


Sessions: 137,033 rows × 11 cols


,Referrer source,Referrer name,Session city,UTM campaign,UTM medium,UTM source,Landing page path,Landing page URL,Online store visitors,Sessions,_source_file
0,search,google,Singapore,NaN,NaN,NaN,/,https://lushprotein.com/,3680,4408,4_1.Sessions by referrer_20260505.csv
1,direct,NaN,NaN,NaN,NaN,NaN,/,https://www.lushprotein.com/,3340,4308,4_1.Sessions by referrer_20260505.csv
2,direct,NaN,Singapore,NaN,NaN,NaN,/,https://lushprotein.com/,2713,3972,4_1.Sessions by referrer_20260505.csv


✅ Saved: bronze_sessions.parquet


## 5. Recharge Data (5 files)

In [9]:
recharge_map = {
    '5_1.orders_combined_20260505.xlsx'      : 'bronze_rc_orders.parquet',
    '5_2.order_items_checkout_20260505.xlsx' : 'bronze_rc_items_checkout.parquet',
    '5_3.subscribers_reactivated_20260505.xlsx': 'bronze_rc_reactivated.parquet',
    '5_4.subscriptions_churned_20260505.xlsx': 'bronze_rc_churned.parquet',
    '5_5.order_items_recurring_20260505.xlsx': 'bronze_rc_items_recurring.parquet',
}

recharge_dfs = {}
for fname, out_name in recharge_map.items():
    df = pd.read_excel(os.path.join(BASE, '5.Recharge_data', fname), dtype=str)
    df['_source_file'] = fname
    key = out_name.replace('bronze_', '').replace('.parquet', '')
    recharge_dfs[key] = df
    df.to_parquet(os.path.join(BRONZE_DIR, out_name), index=False)
    print(f"{fname:55s} → {df.shape[0]:>5,} rows × {df.shape[1]} cols  ✅")


5_1.orders_combined_20260505.xlsx                       → 1,215 rows × 11 cols  ✅
5_2.order_items_checkout_20260505.xlsx                  → 1,094 rows × 15 cols  ✅
5_3.subscribers_reactivated_20260505.xlsx               →    50 rows × 5 cols  ✅
5_4.subscriptions_churned_20260505.xlsx                 →   526 rows × 13 cols  ✅
5_5.order_items_recurring_20260505.xlsx                 →   650 rows × 15 cols  ✅


## 6. Bronze Layer Summary

In [10]:
summary = {
    'bronze_orders'          : bronze_orders.shape,
    'bronze_products'        : bronze_products.shape,
    'bronze_discounts'       : bronze_discounts.shape,
    'bronze_sessions'        : bronze_sessions.shape,
}
for key, df in recharge_dfs.items():
    summary[f'bronze_{key}'] = df.shape

print(f"{'Table':<35} {'Rows':>8}  {'Cols':>6}")
print('-' * 55)
for name, (r, c) in summary.items():
    print(f"{name:<35} {r:>8,}  {c:>6}")

print("\n✅ All bronze tables saved to:", BRONZE_DIR)


Table                                   Rows    Cols
-------------------------------------------------------
bronze_orders                        153,828      70
bronze_products                          167      25
bronze_discounts                         367      18
bronze_sessions                      137,033      11
bronze_rc_orders                       1,215      11
bronze_rc_items_checkout               1,094      15
bronze_rc_reactivated                     50       5
bronze_rc_churned                        526      13
bronze_rc_items_recurring                650      15

✅ All bronze tables saved to: /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/bronze
